In [ ]:
from collections import Counter
from functools import reduce
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from rdkit.Chem import PandasTools
from rdkit.Chem.AllChem import EmbedMolecule
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.rdmolfiles import (
    MolFromMolFile,
    MolFromSmiles,
    MolToMolFile,
    SDMolSupplier,
    SDWriter,
)
from rdkit.Chem.rdmolops import AddHs, CombineMols, SanitizeMol
from rdkit.Chem.rdShapeHelpers import EncodeShape, ShapeProtrudeDist
from rdkit.Geometry.rdGeometry import Point3D, UniformGrid3D
from rdkit.rdBase import DisableLog
from tqdm import tqdm

In [ ]:
tqdm.pandas()

In [ ]:
def add_hs(mol: Mol) -> Mol:
    try:
        assert mol is not None
        SanitizeMol(mol)
        AddHs(mol)
    except:
        return None
    return mol

In [ ]:
file = "/homes/buttensc/Projects/semla-flow/data/unconditional/geom-drugs/train.sdf"
supplier = SDMolSupplier(file, sanitize=True, removeHs=False)
DisableLog("rdApp.*")
df = pd.DataFrame({"mol": [add_hs(mol) for mol in tqdm(supplier) if mol is not None]})
print(f"Loaded {len(df)} molecules")

In [ ]:
df["num_heavy"] = df["mol"].apply(lambda mol: mol.GetNumHeavyAtoms())
df["num_total"] = df["mol"].apply(lambda mol: mol.GetNumAtoms())

In [ ]:
df["atom_types"] = df["mol"].progress_apply(
    lambda mol: Counter(atom.GetSymbol() for atom in mol.GetAtoms())
)

In [ ]:
# combine Counters in atom types
counters = df["atom_types"].tolist()
reduce(lambda x, y: x + y, counters)

In [ ]:
# get box dimensions large enough to enclose ALL molecules
mols = MolFromMolFile("data/conditional_xchem/mpro_active_site_combined.sdf")
pos_all = [mol.GetConformer().GetPositions() for mol in df["mol"]]
pos_all += [mols.GetConformer().GetPositions()]

left_bottom = np.array([p.min(axis=0) for p in pos_all]).min(axis=0)
right_top = np.array([p.max(axis=0) for p in pos_all]).max(axis=0)
print(np.stack([left_bottom, right_top]))
right_top -= left_bottom


In [ ]:
# calculate all volumes on the same grid
# reference: https://github.com/rdkit/rdkit/blob/41a2a79fa87c4f265cd22589fc226f382d380827/Code/GraphMol/ShapeHelpers/ShapeUtils.cpp#L144
def calculate_volume(mol):
    # create empty grid
    grid = UniformGrid3D(*right_top, offSet=Point3D(*left_bottom))

    # encode molecule shape
    EncodeShape(mol, grid, ignoreHs=False)

    # create occupany vector
    vector = grid.GetOccupancyVect()

    # calculate volume
    volume = vector.GetTotalVal()

    return volume

In [ ]:
df["volume"] = df["mol"].progress_apply(calculate_volume)

In [ ]:
mols = mol_mols

In [ ]:
vol_mols = calculate_volume(mols)
print(vol_mols)

In [ ]:
fig = sns.histplot(
    df, x="volume", log_scale=False, element="step", fill=False, bins=100
)
fig.set_title("Volume distribution of molecules in GEOM Training dataset (n = 240 988)")
fig.set_xlabel("Volume")

In [ ]:
fig = sns.regplot(
    data=df,
    x="num_total",
    y="volume",
    order=1,
    ci=80,
    marker=".",
    color=".3",
    line_kws=dict(color="r"),
)
fig.set_title("Volume vs. number of atoms in GEOM Training dataset (n = 240 988)")
fig.set_xlabel("Number of all atoms")
fig.set_ylabel("Volume")


In [ ]:
fig = sns.regplot(
    data=df,
    y="num_total",
    x="volume",
    order=1,
    ci=80,
    marker=".",
    color=".3",
    line_kws=dict(color="r"),
)
fig.set_title("Volume vs. number of atoms in GEOM Training dataset (n = 240 988)")
fig.set_ylabel("Number of atoms")
fig.set_xlabel("Volume")


In [ ]:
model = sm.OLS(df["num_total"], df[["volume"]])
results = model.fit()
results.summary()

In [ ]:
from functools import reduce
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from rdkit.Chem import PandasTools
from rdkit.Chem.AllChem import EmbedMolecule
from rdkit.Chem.rdmolfiles import (
    MolFromMolFile,
    MolFromSmiles,
    MolToMolFile,
    SDMolSupplier,
    SDWriter,
)
from rdkit.Chem.rdmolops import CombineMols
from rdkit.Chem.rdShapeHelpers import EncodeShape, ShapeProtrudeDist
from rdkit.Geometry.rdGeometry import Point3D, UniformGrid3D
from rdkit.rdBase import DisableLog
from tqdm import tqdm

In [ ]:
file = "/homes/buttensc/Projects/semla-flow/data/conditional_xchem/mpro_active_site_fragments_separate.sdf"
# file = "/homes/buttensc/Projects/semla-flow/data/conditional_xchem/mpro_active_site_separate.sdf"
supplier = SDMolSupplier(file, sanitize=False, removeHs=False)
mols_active_site = [
    add_hs(mol)
    for mol in tqdm(supplier)
    if mol is not None and mol.GetNumHeavyAtoms() <= 30
]

In [ ]:
# m = 30.0
# left_bottom = np.array([-m] * 3)
# right_top = np.array([m] * 3)
# right_top -= left_bottom

In [ ]:
# def get_occupancy_vector(mol):
#     grid = UniformGrid3D(*right_top, offSet=Point3D(*left_bottom))
#     EncodeShape(mol, grid, ignoreHs=False)
#     vector = grid.GetOccupancyVect()
#     return np.array(vector)

# np.array([get_occupancy_vector(mol) for mol in tqdm(frags)]).max(axis=0).sum()

# max(calculate_volume(m) for m in frags)

# calculate_volume(mol_frags)

In [ ]:
mols_all = np.array(mols_active_site)
n = len(mols_all)

np.random.seed(42)
rng = np.random.default_rng()

samples = []
volumes = []
num_frags = []
indices = []
choices = set()
for m in tqdm(range(1, 25)):
    list_of_indices = list(combinations(range(n), m))
    if len(list_of_indices) > 1000:
        list_of_indices = rng.choice(list_of_indices, 1000, replace=False)

    # binary_vector = np.random.randint(0, 2, size=n).astype(bool)
    # index = tuple(sorted(rng.choice(n, m, replace=False)))

    for index in list_of_indices:
        frags_filtered = mols_all[np.array(index)]
        frags_combined = reduce(CombineMols, frags_filtered)
        volume = calculate_volume(frags_combined)
        volumes.append(volume)
        num_frags.append(m)
        indices.append(index)

In [ ]:
df_comb = pd.DataFrame({"num_frags": num_frags, "volume": volumes, "indices": indices})
df_comb["num_total_exp"] = df_comb["volume"].apply(
    lambda x: round(results.predict(x)[0])
)

In [ ]:
sns.scatterplot(data=df_comb, x="num_frags", y="volume", alpha=0.1)

In [ ]:
sns.scatterplot(data=df_comb, x="num_frags", y="num_total_exp", alpha=0.1)

In [ ]:
def combine_frags(index):
    frags_filtered = mols_all[np.array(index)]
    frags_combined = reduce(CombineMols, frags_filtered)
    return frags_combined

In [ ]:
# df_sel = df_comb[(df_comb.volume > 4000) & (df_comb.volume < 11000)].copy()
# df_sel["num_heavy"] = df_sel["mol_combined"].apply(lambda mol: mol.GetNumHeavyAtoms())

df_sel = df_comb[
    (df_comb.num_total_exp >= 20)
    & (df_comb.num_frags >= 3)  # & (df_comb.num_frags < 23)
]

In [ ]:
df_sel.groupby("num_total_exp").size()

In [ ]:
df_sel.groupby("num_frags").size()

In [ ]:
# df_sel = df_sel[df_sel.num_frags < 13]
# select first 10 rows per num_frags
df_sel = df_sel.groupby("num_total_exp").head(10).reset_index(drop=True)

In [ ]:
df_sel[["num_frags", "volume", "num_total_exp"]]

In [ ]:
df_sel["mol_combined"] = df_sel["indices"].apply(combine_frags)

In [ ]:
df_sel

In [ ]:
writer = SDWriter("data/conditional_xchem/mpro_active_site_fragments_subsampled.sdf")
for row in tqdm(df_sel.iterrows()):
    mol = row[1]["mol_combined"]
    num_frags = row[1]["num_frags"]
    num_total_exp = row[1]["num_total_exp"]
    volume = row[1]["volume"]
    mol.SetProp("_Name", f"num_frags_{num_frags}_num_total_exp_{num_total_exp}")
    mol.SetProp("num_frags", str(num_frags))
    mol.SetProp("num_total_exp", str(num_total_exp))
    mol.SetProp("volume", str(volume))
    writer.write(mol)